In [ ]:

!pip uninstall -y numpy

!pip install numpy==1.26.4 --quiet
!pip install git+https://github.com/m-bain/whisperX.git --quiet
!pip install pypinyin --quiet


In [3]:
import whisperx
import gc
from pypinyin import pinyin, Style
device = "cuda"
audio_file = "/kaggle/input/datasets/dunbic/chinese-videos/Screen Recording 2026-04-16 112631.mp4"
batch_size = 16 
compute_type = "float16" 

print("--- Bước 1: Đang nhận diện giọng nói (Whisper V3) ---")
model = whisperx.load_model("large-v3", device, compute_type=compute_type)
audio = whisperx.load_audio(audio_file)
result = model.transcribe(audio, batch_size=batch_size)
model = None
gc.collect()

print("--- Bước 2: Đang căn chỉnh thời gian (Aligning) ---")
model_a, metadata = whisperx.load_align_model(language_code="zh", device=device)
result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)

def get_pinyin(text):
    res = pinyin(text, style=Style.TONE)
    return " ".join([item[0] for item in res])

print("\n--- KẾT QUẢ TEST (5 câu đầu) ---")
for segment in result["segments"][:5]:
    cn = segment['text']
    py = get_pinyin(cn)
    print(f"[{segment['start']:.2f}s]: {cn}")
    print(f"PY: {py}\n")

--- Bước 1: Đang nhận diện giọng nói (Whisper V3) ---
2026-04-16 04:42:16 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-04-16 04:42:16 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio

2026-04-16 04:42:21 - whisperx.asr - INFO - Detected language: zh (0.96) in first 30s of audio
--- Bước 2: Đang căn chỉnh thời gian (Aligning) ---


preprocessor_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.28G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.28G [00:00<?, ?B/s]


--- KẾT QUẢ TEST (5 câu đầu) ---
[2.04s]: 可是妈没做到妈要是早跟他离了他今天就不会来找你了妈真的不怪你就算你早跟他离了他今天还是会来他们就是不想让我顺利考试
PY: kě shì mā méi zuò dào mā yào shì zǎo gēn tā lí le tā jīn tiān jiù bú huì lái zhǎo nǐ le mā zhēn de bù guài nǐ jiù suàn nǐ zǎo gēn tā lí le tā jīn tiān hái shì huì lái tā men jiù shì bù xiǎng ràng wǒ shùn lì kǎo shì

[24.68s]: 我宰了夏家这帮王八蛋你砍了他然后呢他进医院你进法院吗那也不能赔了这帮王八蛋哪舅舅你看我像是那种有仇不报的人吗不过不是现在现在考试才是头等大事是啊哥小兰马上要考试了你别再惹事了好了之前是我小乔夏子虞他们一家人的恶毒
PY: wǒ zǎi le xià jiā zhè bāng wáng bā dàn nǐ kǎn le tā rán hòu ne tā jìn yī yuàn nǐ jìn fǎ yuàn ma nà yě bù néng péi le zhè bāng wáng bā dàn nǎ jiù jiù nǐ kàn wǒ xiàng shì nà zhǒng yǒu chóu bù bào de rén ma bù guò bú shì xiàn zài xiàn zài kǎo shì cái shì tóu děng dà shì shì a gē xiǎo lán mǎ shàng yào kǎo shì le nǐ bié zài rě shì le hǎo le zhī qián shì wǒ xiǎo qiáo xià zi yú tā men yī jiā rén de è dú

[51.81s]: 不過現在既然我們有了防備就不會再讓他們得逞了你幹什麼我在熱雞 給小蘭補補媽沒事
PY: bù guò xiàn zài jì rán wǒ men yǒu le fáng bèi jiù bù huì zài ràng tā men dé chěng le nǐ gàn shén me wǒ 

In [5]:
def format_time(s):
    ms = int((s % 1) * 1000)
    s = int(s)
    m, s = divmod(s, 60)
    h, m = divmod(m, 60)
    return f"{h:02}:{m:02}:{s:02},{ms:03}"

with open("nhiem_vu_asr.srt", "w", encoding="utf-8") as f:
    for i, seg in enumerate(result["segments"]):
        f.write(f"{i+1}\n")
        f.write(f"{format_time(seg['start'])} --> {format_time(seg['end'])}\n")
        f.write(f"{seg['text']}\n")
        f.write(f"{get_pinyin(seg['text'])}\n\n")
